In [1]:
from database.adatabase import ADatabase
import pandas as pd
from modeler.modeler import Modeler as m
import matplotlib.pyplot as plt
from processor.processor import Processor as processor
from xgboost.sklearn import XGBRegressor
from tqdm import tqdm
import warnings
warnings.simplefilter(action="ignore")
import pickle
from datetime import datetime, timedelta, timezone
from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor

In [2]:
db = ADatabase("algo")
market = ADatabase("market")
fed = ADatabase("fed")
market.connect()
sp100 = market.retrieve("sp100")
market.disconnect()

In [3]:
training_date = (datetime.now() - timedelta(days=365.25)).astimezone(timezone.utc)
training_years = 4
training_start_date = (training_date-timedelta(days=365.25*training_years)).astimezone(timezone.utc)
holding_period = 5
tickers = sp100["ticker"].values
weeks = 13
factors = [str(i) for i in range(weeks)]
positions = len(sp100["GICS Sector"].unique())

In [ ]:
market.connect()
prices = []
for ticker in tqdm(tickers,desc="model_prep"):
    try:
        ticker_prices = processor.column_date_processing(market.query("prices",{"ticker":ticker}))
        ticker_prices.sort_values("date",inplace=True)
        training_data = ticker_prices[["year","date","ticker","adjclose"]].groupby(["date","ticker"]).mean().reset_index()
        for i in range(weeks):
            training_data[str(i)] = training_data["adjclose"].shift(i)
        training_data["y"] = training_data["adjclose"].shift(-holding_period)
        model_data = training_data[(training_data["date"]<=training_date) & (training_data["date"]>=training_start_date)].dropna().reset_index(drop=True)
        simulation = training_data[training_data["date"]>=training_date]
        # model = XGBRegressor(booster="gblinear",learning_rate=1,objective="reg:absoluteerror",verbosity = 0)
        # model.fit(model_data[factors],model_data["y"])
        # simulation["prediction"] = model.predict(simulation[factors])
        models = m.regression({"X":model_data[factors],"y":model_data["y"]})
        simulation = m.predict(models,simulation,factors)
        ticker_prices = processor.merge(ticker_prices,simulation,on=["year","date","ticker"])
        ticker_prices.sort_values("date",inplace=True)
        ticker_prices["predicted_return"] = (ticker_prices["prediction"] - ticker_prices["adjclose"]) / ticker_prices["adjclose"]
        ticker_prices["signal"] = ticker_prices["predicted_return"]
        ticker_prices["buy_price"] = ticker_prices["adjclose"].shift(-1)
        ticker_prices["buy_date"] = ticker_prices["date"].shift(-1)
        ticker_prices["sell_price"] = ticker_prices["adjclose"].shift(-holding_period)
        ticker_prices["sell_date"] = ticker_prices["date"].shift(-holding_period)
        ticker_prices["abs"] = ticker_prices["signal"].abs()
        ticker_prices["direction"] = ticker_prices["signal"] / ticker_prices["abs"]
        ticker_prices["return"] = (ticker_prices["sell_price"] - ticker_prices["buy_price"]) / ticker_prices ["buy_price"] * (1/positions) * ticker_prices["direction"]
        prices.append(ticker_prices)
    except Exception as e:
        print(str(e))
        continue
market.disconnect()

model_prep:   2%|██▍                                                                                                                         | 2/101 [02:20<1:57:31, 71.23s/it]

In [ ]:
sim = pd.concat(prices).reset_index(drop=True)
sim.sort_values("date",inplace=True)
sim = processor.merge(sim,sp100,on="ticker")

In [ ]:
## backtest
trades = sim[sim["weekday"]==4]
trades = trades[trades["week"] % int(holding_period/5) == 0]
trades = trades[trades["date"]>training_date]
trades = trades.sort_values("abs").groupby(["date","GICS Sector"]).first().reset_index()

In [ ]:
trades = processor.column_date_processing(trades[["date","abs","direction","ticker","GICS Sector","adjclose","return"]])

In [ ]:
portfolio = trades[["date","return"]].groupby("date").sum().reset_index()
portfolio.sort_values("date",inplace=True)
portfolio = portfolio[portfolio["date"]<portfolio["date"].max()]
portfolio["return"] = portfolio["return"] + 1
portfolio["cr"] = portfolio["return"].cumprod()

In [ ]:
fed.connect()
bench = fed.retrieve("sp500")
fed.disconnect()
bench["date"] = pd.to_datetime(bench["date"],utc=True)
bench["value"] = [float(x) for x in bench["value"]]
portfolio = processor.column_date_processing(portfolio)
portfolio = processor.merge(portfolio,bench,on="date")
portfolio.dropna(inplace=True)
portfolio["bcr"] = (portfolio["value"] - portfolio["value"].iloc[0]) / portfolio["value"].iloc[0] + 1

In [ ]:
plt.plot(portfolio["date"].values,portfolio["cr"].values)
plt.plot(portfolio["date"].values,portfolio["bcr"].values)
plt.show()

In [ ]:
recommendations = trades.tail(positions)

In [ ]:
recommendations

In [ ]:
db.connect()
db.drop('portfolios')
db.drop('trades')
db.drop('recommendations')
db.store("portfolio",portfolio)
db.store("trades",trades)
db.store("recommendations",recommendations)
db.disconnect()